In [5]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier


In [79]:
titanic = sns.load_dataset('titanic')

In [80]:
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [81]:
titanic.drop(columns=['class','who','deck','alive','alone','embark_town','adult_male','sibsp','parch'], inplace=True)

In [82]:
titanic.head()

,survived,pclass,sex,age,fare,embarked
0,0,3,male,22.0,7.2500,S
1,1,1,female,38.0,71.2833,C
2,1,3,female,26.0,7.9250,S
3,1,1,female,35.0,53.1000,S
4,0,3,male,35.0,8.0500,S


In [83]:
titanic.drop(columns=['fare'], inplace=True)

In [84]:
titanic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  891 non-null    int64  
 1   pclass    891 non-null    int64  
 2   sex       891 non-null    object 
 3   age       714 non-null    float64
 4   embarked  889 non-null    object 
dtypes: float64(1), int64(2), object(2)
memory usage: 34.9+ KB


In [90]:
X_train,X_test,y_train,y_test = train_test_split(titanic.drop(columns=['survived']),
                                                             titanic['survived'], test_size=0.2, random_state=42)

In [96]:
si_age = SimpleImputer()
si_embark = SimpleImputer(strategy='most_frequent')

In [97]:
#processing null values in age and embark
X_train_age = si_age.fit_transform(X_train[['age']])
X_train_embark = si_embark.fit_transform(X_train[['embarked']])



In [51]:
X_test_age = si_age.transform(X_test[['age']])
X_test_embark = si_embark.transform(X_test[['embarked']])

In [105]:
ohe_sex = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_embark = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [106]:
X_train_sex = ohe_sex.fit_transform(X_train[['sex']])
X_train_embark = ohe_embark.fit_transform(X_train_embark)

In [107]:
X_train_sex

array([[0., 1.],
       [0., 1.],
       [0., 1.],
       ...,
       [0., 1.],
       [1., 0.],
       [0., 1.]])

In [28]:
X_test_sex = ohe_sex.transform(X_test[['sex']])

In [66]:
X_test1_embark.shape

(179, 3)

In [67]:
X_test_embark = ohe_embark.transform(X_test_embark)

In [31]:
X_train_rem = X_train.drop(columns=['sex', 'age','embarked'])

In [37]:
X_test_rem = X_test.drop(columns=['sex', 'age','embarked'])

In [108]:
X_test_rem

,pclass
709,3
439,2
840,3
720,2
39,3
...,...
433,3
773,3
25,3
84,2


In [69]:
X_train_transformed = np.concatenate((X_train_rem, X_train_sex, X_train_age, X_train_embark), axis=1)
X_test_transformed = np.concatenate((X_test_rem, X_test_sex, X_test_age, X_test_embark), axis=1)

In [43]:
clf = DecisionTreeClassifier()

In [44]:
clf.fit(X_train_transformed, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [72]:
y_pred = clf.predict(X_test_transformed)

In [73]:
from sklearn.metrics import accuracy_score

In [74]:
accuracy_score(y_test, y_pred)

0.776536312849162

In [75]:
import pickle

In [77]:
pickle.dump(ohe_sex, open('models/ohe_sex.pkl','wb'))

In [78]:
pickle.dump(ohe_embark, open('models/ohe_embark.pkl','wb'))
pickle.dump(clf, open('models/clf.pkl','wb'))